# Cook County Diabetes and Social Vulnerability

Question: Which Cook County census tracts have both high diabetes prevalence and high social vulnerability?

PLACES 2025 and SVI 2022 are not year-matched. Their ACS periods overlap, and both use 2020-era tract geography. Treat results as a cross-sectional comparison with this data-vintage caveat.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.api as sm

project_dir = Path.cwd()
if not (project_dir / "data" / "raw").exists():
    project_dir = project_dir.parent

raw_dir = project_dir / "data" / "raw"
processed_dir = project_dir / "data" / "processed"
places_path = raw_dir / "places_2025_census_tract.csv"
svi_path = raw_dir / "svi_2022_illinois_tract.csv"
county_fips = "17031"

assert places_path.exists(), f"Missing input file: {places_path}"
assert svi_path.exists(), f"Missing input file: {svi_path}"
print(f"Project directory: {project_dir}")

Project directory: /Users/maiagriffith/Desktop/chicago-health-dashboard-adf


## Load and Filter Sources

Read tract codes as strings so leading zeros are preserved. Only the columns used in this analysis are loaded. Raw source files are never modified.

In [2]:
places_header = pd.read_csv(places_path, nrows=0).columns.str.strip().tolist()
health_measure_source_columns = [
    column for column in places_header if column.endswith("_CrudePrev")
]
places_columns = [
    "StateAbbr", "CountyName", "CountyFIPS", "TractFIPS",
    "DIABETES_Crude95CI", *health_measure_source_columns,
]
svi_columns = [
    "STCNTY", "COUNTY", "FIPS", "RPL_THEMES",
    "RPL_THEME1", "RPL_THEME2", "RPL_THEME3", "RPL_THEME4",
]

places_all = pd.read_csv(
    places_path,
    dtype="string",
    usecols=lambda column: column.strip() in places_columns,
)
places_all.columns = places_all.columns.str.strip()
svi_all = pd.read_csv(svi_path, dtype="string", usecols=svi_columns)

places = places_all.loc[places_all["CountyFIPS"].str.strip() == county_fips].copy()
places["GEOID"] = places["TractFIPS"].str.strip()
for column in health_measure_source_columns:
    places[column] = pd.to_numeric(places[column].str.strip(), errors="raise")
if places[health_measure_source_columns].isna().any().any():
    raise ValueError("PLACES prevalence fields contain missing values; stop before filtering rows.")
places["diabetes_prevalence_pct"] = places["DIABETES_CrudePrev"]

svi = svi_all.loc[svi_all["STCNTY"].str.strip() == county_fips].copy()
svi["GEOID"] = svi["FIPS"].str.strip()
svi_score_columns = ["RPL_THEMES", "RPL_THEME1", "RPL_THEME2", "RPL_THEME3", "RPL_THEME4"]
sentinel_counts = {
    column: int(svi[column].isin(["-999.0", "-999"]).sum())
    for column in svi_score_columns
}
print("PLACES Cook County rows before join:", len(places))
print("SVI Cook County rows before join:", len(svi))
print("PLACES prevalence measures loaded:", len(health_measure_source_columns))
print("SVI -999 sentinel counts:", sentinel_counts)

PLACES Cook County rows before join: 1328
SVI Cook County rows before join: 1331
PLACES prevalence measures loaded: 40
SVI -999 sentinel counts: {'RPL_THEMES': 3, 'RPL_THEME1': 3, 'RPL_THEME2': 3, 'RPL_THEME3': 3, 'RPL_THEME4': 3}


## Validate IDs and Join

The join is one-to-one on the 11-character tract GEOID. SVI's documented sentinel value `-999` is treated as missing, not as a percentile. The outer join retains unmatched records for the audit.

In [3]:
for frame_name, frame in [("PLACES", places), ("SVI", svi)]:
    invalid_ids = frame.loc[~frame["GEOID"].str.fullmatch(r"\d{11}"), "GEOID"].tolist()
    duplicate_ids = frame.loc[frame["GEOID"].duplicated(keep=False), "GEOID"].tolist()
    print(f"{frame_name} invalid GEOIDs ({len(invalid_ids)}):", invalid_ids)
    print(f"{frame_name} duplicate GEOIDs ({len(duplicate_ids)}):", sorted(set(duplicate_ids)))
    assert not invalid_ids, f"Unexpected invalid {frame_name} GEOIDs"
    assert not duplicate_ids, f"Unexpected duplicate {frame_name} GEOIDs"

for column in svi_score_columns:
    svi[column] = svi[column].replace({"-999.0": pd.NA, "-999": pd.NA})
    svi[column] = pd.to_numeric(svi[column], errors="raise")

merged = places[
    ["GEOID", "CountyName", *health_measure_source_columns, "DIABETES_Crude95CI", "diabetes_prevalence_pct"]
].merge(
    svi[["GEOID", "RPL_THEMES", "RPL_THEME1", "RPL_THEME2", "RPL_THEME3", "RPL_THEME4"]],
    on="GEOID",
    how="outer",
    indicator=True,
    validate="one_to_one",
)
print("Rows after outer join:", len(merged))
print("Join status counts:\n", merged["_merge"].value_counts().to_string())
places_only_geoids = sorted(merged.loc[merged["_merge"] == "left_only", "GEOID"].tolist())
svi_only_geoids = sorted(merged.loc[merged["_merge"] == "right_only", "GEOID"].tolist())
print(f"PLACES-only GEOIDs ({len(places_only_geoids)}):", places_only_geoids)
print(f"SVI-only GEOIDs ({len(svi_only_geoids)}):", svi_only_geoids)
print("Matched GEOIDs:", int((merged["_merge"] == "both").sum()))

matched = merged.loc[merged["_merge"] == "both"].copy()
matched["DIABETES_Crude95CI"] = matched["DIABETES_Crude95CI"].astype("string")
model_columns = ["diabetes_prevalence_pct", "RPL_THEME1", "RPL_THEME2", "RPL_THEME3", "RPL_THEME4"]
missing_model_values = matched[model_columns].isna().sum()
print("Missing model values among matched tracts:\n", missing_model_values.to_string())
if missing_model_values.any():
    raise ValueError("Matched analysis data contain missing model values. Stop and review before dropping or imputing.")

PLACES invalid GEOIDs (0): []
PLACES duplicate GEOIDs (0): []
SVI invalid GEOIDs (0): []
SVI duplicate GEOIDs (0): []
Rows after outer join: 1331
Join status counts:
 _merge
both          1328
right_only       3
left_only        0
PLACES-only GEOIDs (0): []
SVI-only GEOIDs (3): ['17031381700', '17031980000', '17031980100']
Matched GEOIDs: 1328
Missing model values among matched tracts:
 diabetes_prevalence_pct    0
RPL_THEME1                 0
RPL_THEME2                 0
RPL_THEME3                 0
RPL_THEME4                 0


## Explore the Matched Tracts

Summaries and plots describe tract-level data; they do not describe individual residents. No observations are imputed.

In [4]:
analysis_columns = ["diabetes_prevalence_pct", "RPL_THEMES", *model_columns[1:]]
print("Summary statistics:\n", matched[analysis_columns].describe().round(3).to_string())

for column, title in [("diabetes_prevalence_pct", "Diabetes prevalence"), ("RPL_THEMES", "Overall SVI percentile")]:
    figure = px.histogram(matched, x=column, nbins=30, title=title)
    figure.show()

correlations = matched[analysis_columns].corr()
print("Correlation matrix:\n", correlations.round(3).to_string())
px.imshow(correlations, text_auto=".2f", color_continuous_scale="RdBu_r", zmin=-1, zmax=1, title="Tract-level correlations").show()

quartile_labels = ["Q1 lowest", "Q2", "Q3", "Q4 highest"]
matched["SVI_quartile"] = pd.qcut(
    matched["RPL_THEMES"].rank(method="first"), q=4, labels=quartile_labels
)
quartile_summary = matched.groupby("SVI_quartile", observed=False).agg(
    tract_count=("GEOID", "count"),
    mean_diabetes_prevalence_pct=("diabetes_prevalence_pct", "mean"),
    median_diabetes_prevalence_pct=("diabetes_prevalence_pct", "median"),
    mean_overall_svi=("RPL_THEMES", "mean"),
).round(3)
quartile_summary

Summary statistics:
        diabetes_prevalence_pct  RPL_THEMES  RPL_THEME1  RPL_THEME2  RPL_THEME3  RPL_THEME4
count                   1328.0      1328.0      1328.0      1328.0      1328.0      1328.0
mean                    12.241       0.596       0.595       0.523       0.688       0.551
std                      4.793       0.288         0.3       0.308       0.228        0.27
min                        2.0       0.002       0.001       0.001       0.121         0.0
25%                        9.1       0.362       0.332       0.258       0.518       0.338
50%                       11.4       0.663       0.667        0.56       0.732       0.559
75%                       15.6       0.844       0.856       0.794       0.888       0.785
max                       32.3       0.999         1.0         1.0       0.983         1.0


Correlation matrix:
                          diabetes_prevalence_pct  RPL_THEMES  RPL_THEME1  RPL_THEME2  RPL_THEME3  RPL_THEME4
diabetes_prevalence_pct                    1.000       0.756       0.753       0.664       0.750       0.112
RPL_THEMES                                 0.756       1.000       0.930       0.685       0.784       0.514
RPL_THEME1                                 0.753       0.930       1.000       0.553       0.822       0.309
RPL_THEME2                                 0.664       0.685       0.553       1.000       0.484       0.052
RPL_THEME3                                 0.750       0.784       0.822       0.484       1.000       0.187
RPL_THEME4                                 0.112       0.514       0.309       0.052       0.187       1.000


,tract_count,mean_diabetes_prevalence_pct,median_diabetes_prevalence_pct,mean_overall_svi
SVI_quartile,,,,
Q1 lowest,332,7.47,7.8,0.182
Q2,332,10.594,10.3,0.524
Q3,332,13.781,12.8,0.756
Q4 highest,332,17.121,17.0,0.923


## Candidate Priority Tracts

Using the approved rule, a candidate is at or above the 75th-percentile cutoff for both diabetes prevalence and overall SVI among the matched Cook County tracts. The flags are screening candidates, not a final outreach decision.

In [5]:
diabetes_cutoff = matched["diabetes_prevalence_pct"].quantile(0.75)
svi_cutoff = matched["RPL_THEMES"].quantile(0.75)
matched["priority_candidate"] = (
    matched["diabetes_prevalence_pct"].ge(diabetes_cutoff)
    & matched["RPL_THEMES"].ge(svi_cutoff)
)
print(f"Diabetes 75th-percentile cutoff: {diabetes_cutoff:.2f}")
print(f"Overall SVI 75th-percentile cutoff: {svi_cutoff:.4f}")
print("Candidate priority tracts:", int(matched["priority_candidate"].sum()))

px.scatter(
    matched, x="RPL_THEMES", y="diabetes_prevalence_pct",
    color="priority_candidate", hover_name="GEOID",
    labels={"RPL_THEMES": "Overall SVI percentile", "diabetes_prevalence_pct": "Diabetes prevalence (%)"},
    title="Diabetes prevalence and overall SVI by tract",
).show()

priority_table = matched.loc[matched["priority_candidate"]].sort_values(
    ["diabetes_prevalence_pct", "RPL_THEMES"], ascending=False
)
priority_table[["GEOID", "diabetes_prevalence_pct", "RPL_THEMES", *model_columns[2:]]].head(20)

Diabetes 75th-percentile cutoff: 15.60
Overall SVI 75th-percentile cutoff: 0.8444
Candidate priority tracts: 214


,GEOID,diabetes_prevalence_pct,RPL_THEMES,RPL_THEME2,RPL_THEME3,RPL_THEME4
475,17031461000,32.3,0.9736,0.9874,0.9015,0.7958
1261,17031835600,29.1,0.9349,0.9407,0.9834,0.7328
1260,17031835500,28.7,0.9653,0.9171,0.9165,0.8679
601,17031670700,26.8,0.9926,0.9822,0.9834,0.7337
615,17031680900,25.3,0.9567,0.7319,0.9475,0.7184
333,17031260100,25.1,0.9435,0.6502,0.9355,0.7715
343,17031270500,25.0,0.9699,0.9742,0.9586,0.7872
619,17031681300,24.7,0.9367,0.8117,0.9672,0.5326
336,17031260400,24.6,0.8934,0.676,0.9834,0.6201
397,17031340600,24.1,0.976,0.9788,0.8849,0.773


## Proposed Linear Regression

Fit the approved unweighted ordinary least squares model with diabetes crude prevalence (%) as the outcome and the four SVI theme percentiles as predictors. Each matched tract is one observation. Report coefficient estimates and 95% confidence intervals. This is an ecological, associational model, not a causal analysis; interpretation should account for correlations among SVI themes and the demographic inputs used in PLACES estimation.

In [6]:
predictors = ["RPL_THEME1", "RPL_THEME2", "RPL_THEME3", "RPL_THEME4"]
design_matrix = sm.add_constant(matched[predictors])
model = sm.OLS(matched["diabetes_prevalence_pct"], design_matrix).fit()
confidence_intervals = model.conf_int(alpha=0.05)
coefficient_table = pd.DataFrame({
    "coefficient": model.params,
    "ci_95_lower": confidence_intervals[0],
    "ci_95_upper": confidence_intervals[1],
    "p_value": model.pvalues,
})
print(f"Observations: {int(model.nobs)}")
print(f"R-squared: {model.rsquared:.3f}")
coefficient_table.round(4)

Observations: 1328
R-squared: 0.703


,coefficient,ci_95_lower,ci_95_upper,p_value
const,2.1445,1.6000,2.6889,0.0000
RPL_THEME1,4.6863,3.7745,5.5980,0.0000
RPL_THEME2,5.1469,4.5903,5.7036,0.0000
RPL_THEME3,7.5892,6.4934,8.6851,0.0000
RPL_THEME4,-1.1044,-1.6620,-0.5467,0.0001


## Limitations

- PLACES prevalence values are model-based estimates and partly use demographic data, so some association with SVI may be built in.
- The analysis is ecological; tract-level patterns do not describe individuals.
- PLACES 2025 and SVI 2022 have different release/reference periods, despite overlapping ACS years and compatible 2020-era tract geography.
- The priority flag is a screening rule and should be interpreted with local knowledge and community input.

In [7]:
processed_columns = {
    "GEOID": "GEOID",
    "CountyName": "county_name",
    "DIABETES_CrudePrev": "diabetes_prevalence_pct",
    "DIABETES_Crude95CI": "diabetes_95ci",
    "RPL_THEMES": "svi_overall_percentile",
    "RPL_THEME1": "svi_theme1_percentile",
    "RPL_THEME2": "svi_theme2_percentile",
    "RPL_THEME3": "svi_theme3_percentile",
    "RPL_THEME4": "svi_theme4_percentile",
}
tracts = matched[list(processed_columns)].rename(columns=processed_columns).copy()
measure_output_columns = {
    source_column: f"{source_column.removesuffix('_CrudePrev').lower()}_prevalence_pct"
    for source_column in health_measure_source_columns
}
for source_column, output_column in measure_output_columns.items():
    tracts[output_column] = matched[source_column].to_numpy()
tracts["priority_candidate"] = matched["priority_candidate"].to_numpy()
processed_dir.mkdir(parents=True, exist_ok=True)
output_path = processed_dir / "tracts.csv"
tracts.to_csv(output_path, index=False)
print(f"Saved {len(tracts)} matched tracts and {len(measure_output_columns)} PLACES measures to {output_path}")
tracts.head()

Saved 1328 matched tracts and 40 PLACES measures to /Users/maiagriffith/Desktop/chicago-health-dashboard-adf/data/processed/tracts.csv


,GEOID,county_name,diabetes_prevalence_pct,diabetes_95ci,svi_overall_percentile,svi_theme1_percentile,svi_theme2_percentile,svi_theme3_percentile,svi_theme4_percentile,access2_prevalence_pct,...,indeplive_prevalence_pct,disability_prevalence_pct,loneliness_prevalence_pct,foodstamp_prevalence_pct,foodinsecu_prevalence_pct,housinsecu_prevalence_pct,shututility_prevalence_pct,lacktrpt_prevalence_pct,emotionspt_prevalence_pct,priority_candidate
0,17031010100,Cook,12.4,"(11.0, 13.8)",0.8504,0.8722,0.2822,0.7152,0.9201,10.9,...,9.1,28.4,35.8,23.6,24.7,20.5,13.1,12.2,28.6,False
1,17031010201,Cook,13.9,"(12.4, 15.4)",0.9343,0.8104,0.7457,0.825,0.957,17.1,...,11.3,33.0,37.4,33.9,34.6,27.2,17.6,16.3,31.1,False
2,17031010202,Cook,12.6,"(11.1, 14.1)",0.9545,0.918,0.5074,0.7793,0.9791,11.9,...,8.2,27.7,33.7,19.9,21.8,17.9,11.1,10.6,26.8,False
3,17031010300,Cook,11.9,"(10.5, 13.2)",0.8132,0.741,0.2423,0.6553,0.977,9.9,...,7.6,26.7,33.5,16.1,18.2,15.3,9.3,9.0,26.1,False
4,17031010400,Cook,7.6,"( 6.7, 8.5)",0.3644,0.5548,0.019,0.4024,0.8044,8.4,...,7.6,24.0,36.6,15.3,17.7,14.7,8.9,9.6,25.5,False


In [8]:
import geopandas as gpd

boundary_path = raw_dir / "cb_2020_17_tract_500k.zip"
assert boundary_path.exists(), f"Missing map boundary file: {boundary_path}"

cook_boundaries = gpd.read_file(f"zip://{boundary_path}")
cook_boundaries["GEOID"] = cook_boundaries["GEOID"].astype("string").str.strip()
cook_boundaries = cook_boundaries.loc[
    cook_boundaries["GEOID"].str.startswith(county_fips), ["GEOID", "geometry"]
].to_crs("EPSG:4326")
assert not cook_boundaries["GEOID"].duplicated().any()
print("Cook County map polygons:", len(cook_boundaries))

geojson_path = processed_dir / "cook_county_tracts.geojson"
cook_boundaries.to_file(geojson_path, driver="GeoJSON", index=False)
print(f"Saved map geometry to {geojson_path}")

Cook County map polygons: 1331
Saved map geometry to /Users/maiagriffith/Desktop/chicago-health-dashboard-adf/data/processed/cook_county_tracts.geojson
